In [1]:
import pandas as pd
from pandas.api.types import is_string_dtype

def format_ship_dataset(df, survived_col, sex_col, age_col, class_col=None, crew_col=None):
    """
    Trasforma i dataset navali in un formato uniforme:
    ['survived', 'sex', 'age', 'age_missing', 'class', 'crew']
    """
    
    df_new = pd.DataFrame()

    # --- Sopravvivenza ---
    if is_string_dtype(df[survived_col]):
        # Se dtype è string allora converto a numero
        df_new['survived'] = df[survived_col].str.lower().map({'lost': 0, 'saved': 1})
    else:
        df_new['survived'] = df[survived_col].astype(int)

    # --- Sesso ---
    # un solo attributo per il sesso e non due di cui uno derivabile dal primo (ad esempio con male e female avremmo female = 1 - male)
    df_new['sex'] = df[sex_col].fillna("").str.lower().isin(['female', 'f']).astype(int) 

    # --- Età ---
    median_age = df[age_col].median() # se l'età non c'è si prende l'età mediana
    df_new['age'] = df[age_col].fillna(median_age)
    df_new["age_missing"] = df[age_col].isna().astype(int)

    # --- Classe ---
    if class_col is not None:
        df_new['class'] = df[class_col].fillna(0).astype(int)
    else:
        df_new['class'] = 0

    # --- Crew ---
    if crew_col is not None:
        df_new['crew'] = df[crew_col].map({'P': 0, 'Passenger': 0, 'C': 1, 'Crew': 1}).fillna(-1).astype(int)
    else:
        df_new['crew'] = -1

    return df_new

In [2]:
# Caricamento dei CSV
df_titanic = pd.read_csv("../data/titanic.csv")
df_estonia = pd.read_csv("../data/estonia.csv")
df_lusitania = pd.read_csv("../data/lusitania.csv")

print(f"Titanic: {df_titanic.shape}")
print(f"Estonia: {df_estonia.shape}")
print(f"Lusitania: {df_lusitania.shape}")

Titanic: (891, 12)
Estonia: (989, 8)
Lusitania: (1961, 16)


In [3]:
df_titanic_formatted = format_ship_dataset(df_titanic, survived_col='Survived', sex_col='Sex', age_col='Age', class_col='Pclass')

df_estonia_formatted = format_ship_dataset(df_estonia, survived_col='Survived', sex_col='Sex', age_col='Age', crew_col='Category')

df_lusitania_formatted = format_ship_dataset(df_lusitania, survived_col='Fate', sex_col='Sex', age_col='Age', crew_col='Passenger/Crew')

print(df_titanic_formatted.head(5))
print(df_estonia_formatted.head(5))
print(df_lusitania_formatted.head(5))

   survived  sex   age  age_missing  class  crew
0         0    0  22.0            0      3    -1
1         1    1  38.0            0      1    -1
2         1    1  26.0            0      3    -1
3         1    1  35.0            0      1    -1
4         0    0  35.0            0      3    -1
   survived  sex  age  age_missing  class  crew
0         0    0   62            0      0     0
1         0    1   22            0      0     1
2         0    1   21            0      0     1
3         0    0   53            0      0     1
4         0    1   55            0      0     0
   survived  sex   age  age_missing  class  crew
0         0    0  38.0            0      0     1
1         0    0  37.0            0      0     1
2         1    0  30.0            0      0     1
3         1    0  25.0            0      0     1
4         1    0  27.0            0      0     1


In [4]:
all_ships = pd.concat([df_titanic_formatted, df_lusitania_formatted, df_estonia_formatted], ignore_index=True)
print(all_ships)

all_ships.to_csv("../data/all_ships.csv", index=False)

      survived  sex   age  age_missing  class  crew
0            0    0  22.0            0      3    -1
1            1    1  38.0            0      1    -1
2            1    1  26.0            0      3    -1
3            1    1  35.0            0      1    -1
4            0    0  35.0            0      3    -1
...        ...  ...   ...          ...    ...   ...
3836         0    1  60.0            0      0     0
3837         1    0  34.0            0      0     0
3838         0    0  77.0            0      0     0
3839         0    1  87.0            0      0     0
3840         1    0  42.0            0      0     0

[3841 rows x 6 columns]
